# Anomaly Detection & Risk Identification

This notebook demonstrates automated statistical anomaly detection and alert monitoring:
1. **Threshold-Based Alerts** checking current metrics against min/max business rules.
2. **Statistical Z-Score Anomaly Detection** ($z > 2$) over a 30-day lookback window.
3. **Severity Classification** (`CRITICAL`, `HIGH`, `MEDIUM`, `LOW`).
4. **Persistent Audit Trail Logging** (`anomalies_log.csv`).
5. **Time-Series Visualization** with 7-day rolling average, shaded $\pm 2\sigma$ expected range, and highlighted anomaly markers.

In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

filepath = '../data/raw/daily_business_metrics.csv'
if not os.path.exists(filepath):
    filepath = 'data/raw/daily_business_metrics.csv'

df = pd.read_csv(filepath)
df['date'] = pd.to_datetime(df['date'])
df_ts = df.set_index('date')
print(f"Loaded 90 days of daily business metrics from {df['date'].min().strftime('%Y-%m-%d')} to {df['date'].max().strftime('%Y-%m-%d')}.")

## Task 1: Threshold-Based Anomaly Detection

In [2]:
alert_rules = {
    'daily_revenue': {'min': 5000, 'max': 50000},
    'transaction_count': {'min': 100, 'max': 10000},
    'signup_rate': {'min': 10, 'max': 500}
}

def check_thresholds(metrics, rules):
    alerts = []
    for metric_name, rule in rules.items():
        value = metrics[metric_name]
        if value < rule['min']:
            alerts.append({'metric': metric_name, 'value': value, 'threshold': rule['min'], 'direction': 'BELOW_MIN', 'severity': 'HIGH'})
        elif value > rule['max']:
            alerts.append({'metric': metric_name, 'value': value, 'threshold': rule['max'], 'direction': 'ABOVE_MAX', 'severity': 'MEDIUM'})
    return alerts

test_metrics = {'daily_revenue': 2500, 'transaction_count': 50, 'signup_rate': 5}
alerts = check_thresholds(test_metrics, alert_rules)
for alert in alerts:
    print(f"ALERT: {alert['metric']} {alert['direction']}: Value={alert['value']} (Threshold={alert['threshold']})")

## Task 2: Statistical Anomaly Detection with Z-Score

In [3]:
daily_revenue = df_ts['amount'].tail(30)
mean = daily_revenue.mean()
std = daily_revenue.std()
z_scores = np.abs((daily_revenue - mean) / std)
anomalies = daily_revenue[z_scores > 2]

print(f"Detected {len(anomalies)} statistical anomalies out of {len(daily_revenue)} days:")
for date, value in anomalies.items():
    print(f"  - {date.strftime('%Y-%m-%d')}: ${value:,.2f} (Z-Score: {z_scores[date]:.2f})")

## Task 3: Severity Classification & Filtering

In [4]:
def classify_severity(value, mean, std):
    z_score = abs((value - mean) / std)
    if z_score > 3.0:
        return 'CRITICAL'
    elif z_score > 2.0:
        return 'HIGH'
    elif z_score > 1.5:
        return 'MEDIUM'
    else:
        return 'LOW'

severity_records = []
for date, value in anomalies.items():
    severity_records.append({
        'date': date.strftime('%Y-%m-%d'),
        'value': value,
        'z_score': round(z_scores[date], 2),
        'severity': classify_severity(value, mean, std)
    })

severity_df = pd.DataFrame(severity_records)
print(severity_df)
print(f"\nCritical/High Alerts: {len(severity_df[severity_df['severity'].isin(['CRITICAL', 'HIGH'])])}")

## Task 4: Anomaly Audit Logging & Persistence

In [5]:
anomaly_log = []
now_ts = pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S')
for date, value in anomalies.items():
    anomaly_log.append({
        'timestamp': now_ts,
        'anomaly_date': date.strftime('%Y-%m-%d'),
        'metric': 'daily_revenue',
        'value': round(value, 2),
        'expected_range': f"${mean-2*std:,.0f} - ${mean+2*std:,.0f}",
        'z_score': round(z_scores[date], 2),
        'severity': classify_severity(value, mean, std),
        'status': 'OPEN'
    })

log_df = pd.DataFrame(anomaly_log)
print(log_df)

## Task 5: Time-Series Visualization with Flagged Anomaly Points

In [6]:
fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(daily_revenue.index, daily_revenue.values, marker='o', label='Daily Revenue ($)', color='#2563eb')
ax.plot(daily_revenue.rolling(window=7).mean().index, daily_revenue.rolling(window=7).mean().values, label='7-Day Moving Average', color='#16a34a', linestyle='--')

for date, value in anomalies.items():
    ax.scatter(date, value, color='#dc2626', s=180, marker='X', zorder=5)
    ax.annotate(f"ANOMALY\n${value:,.0f}", (date, value), xytext=(0, 12), textcoords='offset points', ha='center', fontweight='bold', color='#dc2626')

ax.fill_between(daily_revenue.index, mean - 2 * std, mean + 2 * std, alpha=0.15, color='#3b82f6', label=r'Expected Range (\mu \pm 2\sigma)')
ax.set_title('Daily Revenue Time-Series with Statistical Anomalies Flagged')
ax.set_ylabel('Revenue ($)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.show()